<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MOSS-TTS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MOSS-TTS v1.5 — Multilingual TTS with Pinyin, IPA, and Inline Pause Markers

8B-parameter open-weights TTS by the OpenMOSS team, **state-of-the-art on Seed-TTS-eval** among open-source models. 31 languages, zero-shot voice cloning, long-form generation, plus the richest set of inline controls in the suite: **Pinyin/IPA input**, inline `[pause X.Ys]` markers, language tags, and token-level duration control. Apache 2.0.

### Quick Start
> Requires **Colab Runtime Version 2026.01** (ships torch 2.9.0, MOSS-TTS needs 2.9.1 — compatible). Set via Runtime - Change runtime type - Runtime Version.

1. Connect to a GPU runtime (L4 or A100 recommended - 8B in bf16 needs ~16 GB)
2. Run Steps 1\u20134 in order - no token, no sign-up needed
3. Open the Gradio UI link from Step 4 and start generating

| GPU | VRAM | Status |
|-----|------|--------|
|-----|------|--------|
| A100 | 40 GB | Recommended (fastest) |
|-----|------|--------|
| L4 | 24 GB | Works (recommended tier) |
|-----|------|--------|
| T4 | 16 GB | X not enough VRAM for the 8B weights |

### Generation Tips (from the OpenMOSS team)
- **Use the language tag** for non-Chinese/English inputs: `processor.build_user_message(text=..., language="French")`
- **Inline pause control** - drop `[pause X.Ys]` into your text for precise timing: `it was the name of [pause 3.2s] Quiet Night Thoughts`
- **Pinyin/IPA input** for tricky pronunciation - useful for proper nouns, technical terms, tonal languages
- **Voice cloning** - pass a `reference=[audio_path]` to `build_user_message`. v1.5 handles long reference + short text better than 1.0
- **Duration control** - set `tokens=N` to target a specific output length (in audio tokens)

### Supported Languages (31)
Chinese, English, Cantonese, Arabic, Czech, Danish, Dutch, Finnish, French, German, Greek, Hebrew, Hindi, Hungarian, Italian, Japanese, Korean, Macedonian, Malay, Persian, Polish, Portuguese, Romanian, Russian, Spanish, Swahili, Swedish, Tagalog, Thai, Turkish, Vietnamese.

**License:** Apache 2.0.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones MOSS-TTS and installs all deps. Requires Colab Runtime Version 2026.01.

import os, sys, subprocess
import torch

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime -> Change runtime type -> L4 or A100')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB')

if vram_gb < 20:
    raise SystemExit(
        f'{vram_gb:.0f} GB VRAM is not enough. MOSS-TTS v1.5 in bf16 needs ~16 GB for weights. '
        'Reconnect to a runtime with >=24 GB (L4 or A100).'
    )


# ── numpy._core.umath patch ──────────────────────────────────────────────────
# Colab Runtime 2026.01 ships numpy 2.0.x where the string ufuncs (_center,
# _ljust, _rjust, _zfill, _strip_*, _lstrip_*, _rstrip_*, _partition*,
# _rpartition*, _slice, _expandtabs*, _replace, is*, find/index, etc.) are
# NOT yet exposed in numpy._core.umath. The first import of numpy._core.strings
# (triggered by `import torchaudio`, `import librosa`, `import soundfile`,
# MOSS-TTS' AutoProcessor, or even `np.strings`) then fails with:
#   ImportError: cannot import name '_center' from 'numpy._core.umath'
# MOSS-TTS pulls all of these as transitive deps, so a fresh runtime is
# unusable. We inject minimal pure-Python ufuncs into umath before any of
# those imports happen, so the strings.py `from numpy._core.umath import ...`
# succeeds. MOSS-TTS itself never actually CALLS any of these on string
# arrays during inference — we only need them to *exist* at import time.
import numpy as np
import numpy._core.umath as _umath

_NEEDED = (
    '_center', '_expandtabs', '_expandtabs_length', '_ljust',
    '_lstrip_chars', '_lstrip_whitespace', '_partition',
    '_partition_index', '_replace', '_rjust', '_rpartition',
    '_rpartition_index', '_rstrip_chars', '_rstrip_whitespace',
    '_slice', '_strip_chars', '_strip_whitespace', '_zfill',
    'count', 'endswith', 'find', 'index', 'isalnum', 'isalpha',
    'isdecimal', 'isdigit', 'islower', 'isnumeric', 'isspace',
    'istitle', 'isupper', 'rfind', 'rindex', 'startswith', 'str_len',
)

# Pure-Python wrappers that match the public ufunc semantics closely enough
# for the strings.py re-exports. They use Python's str methods, not np.char,
# to avoid the chicken-and-egg: importing np.char itself triggers strings.py.
def _str_center(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).center(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_ljust(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).ljust(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_rjust(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).rjust(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_zfill(a, width):
    width = int(width)
    return np.array([str(s).zfill(width) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_strip(a, chars=None):
    return np.array([str(s).strip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_lstrip(a, chars=None):
    return np.array([str(s).lstrip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_rstrip(a, chars=None):
    return np.array([str(s).rstrip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _bool_pred(name):
    return lambda a: np.array([getattr(str(s), name)() for s in np.asarray(a).flat],
                              dtype=bool).reshape(np.asarray(a).shape)

_REAL = {
    '_center':           (_str_center, 3, 1),
    '_ljust':            (_str_ljust, 3, 1),
    '_rjust':            (_str_rjust, 3, 1),
    '_zfill':            (_str_zfill, 2, 1),
    '_strip_chars':      (_str_strip, 2, 1),
    '_lstrip_chars':     (_str_lstrip, 2, 1),
    '_rstrip_chars':     (_str_rstrip, 2, 1),
    '_strip_whitespace': (_str_strip, 1, 1),
    '_lstrip_whitespace':(_str_lstrip, 1, 1),
    '_rstrip_whitespace':(_str_rstrip, 1, 1),
    'isalnum':           (_bool_pred('isalnum'),   1, 1),
    'isalpha':           (_bool_pred('isalpha'),   1, 1),
    'isdigit':           (_bool_pred('isdigit'),   1, 1),
    'islower':           (_bool_pred('islower'),   1, 1),
    'isupper':           (_bool_pred('isupper'),   1, 1),
    'isspace':           (_bool_pred('isspace'),   1, 1),
    'isnumeric':         (_bool_pred('isnumeric'), 1, 1),
    'isdecimal':         (_bool_pred('isdecimal'), 1, 1),
    'istitle':           (_bool_pred('istitle'),   1, 1),
}
# Names without a real wrapper get a passthrough ufunc (1 in, 1 out) so the
# import succeeds even if the semantics aren't perfect. None of MOSS-TTS'
# code paths actually call these on string arrays during inference.
_PASSTHROUGH = (1, 1)
_patched = []
for _name in _NEEDED:
    if hasattr(_umath, _name):
        continue
    try:
        _fn, _nin, _nout = _REAL.get(_name, (lambda *a: a[0] if a else '', *_PASSTHROUGH))
        _umath.__dict__[_name] = np.frompyfunc(_fn, _nin, _nout)
        _patched.append(_name)
    except Exception as _e:
        print(f'[numpy_patch] could not patch {_name}: {_e}')

if _patched:
    print(f'[numpy_patch] Injected {len(_patched)} string ufunc(s) into numpy._core.umath: {_patched}')
    print('[numpy_patch] This is a workaround for Colab Runtime 2026.01 numpy 2.0.x where these ufuncs are not yet in umath. Safe to ignore on newer numpy.')
else:
    print('[numpy_patch] numpy._core.umath already has all string ufuncs; no patching needed.')

del _NEEDED, _REAL, _PASSTHROUGH, _patched
if '_fn' in dir(): del _fn
if '_nin' in dir(): del _nin
if '_nout' in dir(): del _nout
if '_name' in dir(): del _name
# ─────────────────────────────────────────────────────────────────────────────

# Clone MOSS-TTS
REPO_DIR = '/content/MOSS-TTS'
REPO_URL = 'https://github.com/OpenMOSS/MOSS-TTS.git'
if not os.path.isdir(REPO_DIR):
    print(f'[git] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.')

# System dep: ffmpeg
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=False)

# Let MOSS-TTS pull numpy 2.x. Upgrade librosa/soundfile for numpy 2.x compat.
# Pin transformers==4.48.0 — MOSS-TTS v1.5 was built for 4.x
# 5.0.0 is baked into Colab's filesystem. Strategy: rm -rf, clean install,
# then --no-deps editable install to avoid pip resolution cascade.
import site, shutil, glob
_sd = site.getsitepackages()[0]

# Step A: Nuke all transformers from site-packages
print('[pip] Nuking all transformers from site-packages ...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'transformers'], check=False)
for pattern in ['transformers', 'transformers-*.dist-info']:
    for p in glob.glob(os.path.join(_sd, pattern)):
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
            print(f'  Removed {os.path.basename(p)}')

# Step B: Clean install 4.48.0 BEFORE touching MOSS-TTS
print('[pip] Installing transformers==4.48.0 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'transformers==4.48.0'], check=True)

# Step C: --no-deps MOSS-TTS editable install (prevents pip resolution cascade to 5.0.0)
print('[pip] Installing MOSS-TTS --no-deps (editable) ...')
subprocess.run(['pip', 'install', '-q', '--no-deps', '-e', f'{REPO_DIR}'], check=True)

# Step D: Manually install MOSS-TTS runtime deps (no transformers conflict risk)
print('[pip] Installing MOSS-TTS runtime dependencies ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate', 'safetensors', 'tokenizers', 'sentencepiece', 'soundfile',
    'librosa', 'numba', 'hf_transfer', 'einops', 'peft', 'torchcodec>=0.3.0'],
    check=False)

# Step E: Verify
import transformers
assert transformers.__version__ == '4.48.0', f'Expected 4.48.0, got {transformers.__version__}'
from transformers.generation import GenerationMixin  # noqa
print(f'[pip] transformers=={transformers.__version__} clean. GenerationMixin OK.')

# UI + extras — upgrade to numpy 2.x compatible versions
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'librosa', 'soundfile', 'numba'], check=True)
subprocess.run(['pip', 'install', '-q', 'gradio==5.33.0', 'tqdm==4.66.5', 'hf_transfer'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'[sys.path] Added {REPO_DIR}')

os.makedirs('/content/audio_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/audio_out and /content/ref_audio ready.')


# Verify install (use importlib.metadata — import transformers would load 5.0.0)
import torch, importlib
print(f'[Verify] torch={torch.__version__} cuda={torch.cuda.is_available()}')
print(f'[Verify] transformers={importlib.metadata.version("transformers")}')
print('[Done] STEP 1 complete. Proceed to Step 2.')

In [ ]:
# @title STEP 2 — Pre-cache Models
# @markdown Downloads MOSS-TTS-v1.5 weights (~16 GB) + 2 reference clips to local cache.
# @markdown Google Drive doesn't support symlinks required by newer huggingface_hub — using local cache for this session.

import os, urllib.request
from huggingface_hub import snapshot_download

# Disable Drive caching for MOSS-TTS (huggingface_hub v1.x needs symlinks)
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# Download MOSS-TTS-v1.5 checkpoint (~16 GB)
REPO = 'OpenMOSS-Team/MOSS-TTS-v1.5'
print(f'[Download] {REPO} (~16 GB) ...')
snapshot_download(REPO)
print('[Download] MOSS-TTS-v1.5 weights cached.')

# Pre-cache two reference clips from OpenMOSS CDN
REFS = {
    'reference_zh.wav': 'https://speech-demo.oss-cn-shanghai.aliyuncs.com/moss_tts_demo/tts_readme_demo/reference_zh.wav',
    'reference_en.m4a': 'https://speech-demo.oss-cn-shanghai.aliyuncs.com/moss_tts_demo/tts_readme_demo/reference_en.m4a',
}
for name, url in REFS.items():
    out_path = f'/content/ref_audio/{name}'
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f'  OK {name} (cached)')
        continue
    print(f'  > {name}')
    urllib.request.urlretrieve(url, out_path)
print('[Download] Reference clips ready.')


In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers. Model is loaded on first use, then stays in memory.

import os, time, tempfile, importlib.util, shutil
import numpy as np
import torch
import soundfile as sf
import torchaudio

# Clear stale HF remote-code cache (may reference old transformers APIs)
_cache = os.path.join(os.path.expanduser('~'), '.cache', 'huggingface', 'modules')
if os.path.isdir(_cache):
    for entry in os.listdir(_cache):
        if 'OpenMOSS' in entry or 'MOSS' in entry:
            p = os.path.join(_cache, entry)
            if os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)
                print(f'[Cache] Removed stale {entry}')


REPO_DIR = '/content/MOSS-TTS'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

REPO_ID = 'OpenMOSS-Team/MOSS-TTS-v1.5'
_PROCESSOR = None
_MODEL = None
_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Backends to try, in order. The cuDNN SDPA backend is broken in some torch builds — disable it.
def _resolve_attn_impl(dtype):
    if _DEVICE != 'cuda':
        return 'eager'
    if (importlib.util.find_spec('flash_attn') is not None
            and dtype in {torch.float16, torch.bfloat16}
            and torch.cuda.get_device_capability()[0] >= 8):
        return 'flash_attention_2'
    return 'sdpa'

def _get_models():
    global _PROCESSOR, _MODEL
    if _PROCESSOR is not None and _MODEL is not None:
        return _PROCESSOR, _MODEL
    from transformers import AutoModel, AutoProcessor
    dtype = torch.bfloat16 if _DEVICE == 'cuda' else torch.float32
    print(f'[Load] {REPO_ID} ({dtype}, {_resolve_attn_impl(dtype)}) ...')
    t0 = time.time()
    _PROCESSOR = AutoProcessor.from_pretrained(REPO_ID, trust_remote_code=True)
    _MODEL = AutoModel.from_pretrained(
        REPO_ID,
        trust_remote_code=True,
        attn_implementation=_resolve_attn_impl(dtype),
        torch_dtype=dtype,
    ).to(_DEVICE).eval()
    # The audio tokenizer lives in the processor but is a torch module — move to device too
    if hasattr(_PROCESSOR, 'audio_tokenizer') and _PROCESSOR.audio_tokenizer is not None:
        _PROCESSOR.audio_tokenizer = _PROCESSOR.audio_tokenizer.to(_DEVICE)
    print(f'[Load] Ready in {time.time()-t0:.1f}s — sample_rate={_PROCESSOR.model_config.sampling_rate} Hz')
    return _PROCESSOR, _MODEL

def _materialize_ref_audio(ref_audio_path):
    """Gradio gives (sr, np.ndarray) for uploads. Save to a temp .wav the model can read."""
    if isinstance(ref_audio_path, tuple) and len(ref_audio_path) == 2:
        sr, audio = ref_audio_path
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        sf.write(tmp.name, audio, sr, format='WAV')
        tmp.flush()
        return tmp.name
    return ref_audio_path  # already a file path

def synth(text: str,
          language: str = '',
          ref_audio_path=None,
          tokens: int | None = None,
          max_new_tokens: int = 4096,
          out_name: str = 'moss.wav') -> tuple[str, int]:
    """Run a single MOSS-TTS generation. Returns (path, sample_rate)."""
    processor, model = _get_models()
    text = (text or '').strip()
    if not text:
        raise RuntimeError('Text is empty.')

    kwargs = dict(text=text)
    if language and language != 'Auto':
        kwargs['language'] = language
    if ref_audio_path:
        kwargs['reference'] = [_materialize_ref_audio(ref_audio_path)]
    if tokens is not None and int(tokens) > 0:
        kwargs['tokens'] = int(tokens)

    user_msg = processor.build_user_message(**kwargs)
    batch = processor([[user_msg]], mode='generation')
    input_ids = batch['input_ids'].to(_DEVICE)
    attention_mask = batch['attention_mask'].to(_DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=int(max_new_tokens),
        )

    sr = processor.model_config.sampling_rate
    out_path = os.path.join('/content/audio_out', out_name)
    sample_idx = 0
    for message in processor.decode(outputs):
        audio = message.audio_codes_list[0]
        if sample_idx == 0:
            torchaudio.save(out_path, audio.unsqueeze(0).cpu(), sr)
        else:
            torchaudio.save(out_path.replace('.wav', f'_{sample_idx}.wav'), audio.unsqueeze(0).cpu(), sr)
        sample_idx += 1
    return out_path, sr

print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app. Pick a language, optionally upload a reference clip, type text, hit Generate. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

LANGUAGES = [
    ('Auto', ''),
    ('Chinese (zh)', 'Chinese'),
    ('English (en)', 'English'),
    ('Cantonese (yue)', 'Cantonese'),
    ('Arabic (ar)', 'Arabic'),
    ('Czech (cs)', 'Czech'),
    ('Danish (da)', 'Danish'),
    ('Dutch (nl)', 'Dutch'),
    ('Finnish (fi)', 'Finnish'),
    ('French (fr)', 'French'),
    ('German (de)', 'German'),
    ('Greek (el)', 'Greek'),
    ('Hebrew (he)', 'Hebrew'),
    ('Hindi (hi)', 'Hindi'),
    ('Hungarian (hu)', 'Hungarian'),
    ('Italian (it)', 'Italian'),
    ('Japanese (ja)', 'Japanese'),
    ('Korean (ko)', 'Korean'),
    ('Macedonian (mk)', 'Macedonian'),
    ('Malay (ms)', 'Malay'),
    ('Persian (fa)', 'Persian'),
    ('Polish (pl)', 'Polish'),
    ('Portuguese (pt)', 'Portuguese'),
    ('Romanian (ro)', 'Romanian'),
    ('Russian (ru)', 'Russian'),
    ('Spanish (es)', 'Spanish'),
    ('Swahili (sw)', 'Swahili'),
    ('Swedish (sv)', 'Swedish'),
    ('Tagalog (tl)', 'Tagalog'),
    ('Thai (th)', 'Thai'),
    ('Turkish (tr)', 'Turkish'),
    ('Vietnamese (vi)', 'Vietnamese'),
]
LANG_CODES = [code for _, code in LANGUAGES]
LANG_DISPLAY = [name for name, _ in LANGUAGES]

def _err(msg):
    return None, f'\u274c {msg}'

def ui_synth(text, language_display, ref_audio, tokens, max_new_tokens, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    try:
        progress(0.1, desc='Loading model (first run is slow)...')
        _get_models()
        progress(0.4, desc='Generating...')
        lang_code = dict(LANGUAGES)[language_display]
        path, sr = synth(
            text.strip(),
            language=lang_code,
            ref_audio_path=ref_audio,
            tokens=tokens if tokens and tokens > 0 else None,
            max_new_tokens=int(max_new_tokens),
            out_name='ui.wav',
        )
        progress(1.0, desc='Done.')
        return path, f'\u2705 Generated @ {sr} Hz'
    except Exception as e:
        return None, f'\u274c {e}'

EXAMPLES = [
    {
        'name': 'Chinese (no ref) — long emotional passage',
        'text': '\u4eb2\u7231\u7684\u4f60\uff0c\u4f60\u597d\u5440\u3002\n\n\u4eca\u5929\uff0c\u6211\u60f3\u7528\u6700\u8ba4\u771f\u3001\u6700\u6e29\u67d4\u7684\u58f0\u97f3\uff0c\u5bf9\u4f60\u8bf4\u4e00\u4e9b\u91cd\u8981\u7684\u8bdd\u3002',
        'lang': 'Chinese (zh)',
        'ref': None,
    },
    {
        'name': 'English (no ref) — AI-era narration',
        'text': 'We stand on the threshold of the AI era. A new era, jointly shaped by humans and intelligent systems, has arrived.',
        'lang': 'English (en)',
        'ref': None,
    },
    {
        'name': 'French (no ref) — with language tag',
        'text': 'Bonjour, je voudrais essayer une voix fran\u00e7aise naturelle et stable.',
        'lang': 'French (fr)',
        'ref': None,
    },
    {
        'name': 'Chinese — inline [pause X.Ys] marker',
        'text': '\u6211\u4eca\u5929\u5b66\u4e60\u4e86\u4e00\u9996\u4e2d\u56fd\u7684\u53e4\u8bd7\uff0c\u5b83\u7684\u540d\u5b57\u662f[pause 3.2s]\u9759\u591c\u601d\uff01',
        'lang': 'Chinese (zh)',
        'ref': None,
    },
    {
        'name': 'English with Chinese reference voice (voice cloning)',
        'text': 'I am solving the equation: x equals negative b, plus or minus the square root of b squared minus four a c, all divided by two a.',
        'lang': 'English (en)',
        'ref': '/content/ref_audio/reference_zh.wav',
    },
    {
        'name': 'Chinese with English reference voice (cross-language clone)',
        'text': '\u4eca\u5929\u7684\u5929\u6c14\u771f\u7684\u5f88\u597d\uff0c\u9002\u5408\u51fa\u95e8\u8d70\u8d70\u3002',
        'lang': 'Chinese (zh)',
        'ref': '/content/ref_audio/reference_en.m4a',
    },
    {
        'name': 'Pinyin input — explicit tone control',
        'text': 'nin2 hao3\uff0cqing3 wen4 nin2 lai2 zi4 na3 zuo4 cheng2 shi4\uff1f',
        'lang': 'Chinese (zh)',
        'ref': None,
    },
    {
        'name': 'IPA input — explicit pronunciation control',
        'text': '/h\u0259lo\u028a, me\u026a a\u026a \u00e6sk w\u026at\u0283 s\u026ati ju\u02d0 \u0251\u02d0r fr\u028cm?/',
        'lang': 'English (en)',
        'ref': None,
    },
]

with gr.Blocks(title='MOSS-TTS v1.5', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# MOSS-TTS v1.5\n**Model:** `OpenMOSS-Team/MOSS-TTS-v1.5` (8B, bf16) \u00b7 **GPU:** {gpu_name} ({vram_gb:.0f} GB)\n\nState-of-the-art open-source TTS. 31 languages, voice cloning, **Pinyin/IPA input**, inline `[pause X.Ys]` markers, language tags, duration control.')
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                label='Text to synthesize',
                value=EXAMPLES[0]['text'],
                lines=5,
                placeholder='Type in any of 31 supported languages. For inline pause, drop [pause X.Ys] into the text.',
            )
            with gr.Row():
                language = gr.Dropdown(LANG_DISPLAY, value='Auto', label='Language (recommended for non-EN/ZH)')
            with gr.Accordion('Voice cloning (optional)', open=False):
                ref_audio = gr.Audio(
                    label='Reference voice (5\u201330s clip)',
                    type='filepath', info="5-30s clean clip. v1.5 handles long-ref + short-text cloning better than v1.0.",
                    )
                gr.Markdown('\ud83d\udc4b\ufe0f v1.5 handles long-reference + short-text cloning better than 1.0.')
            with gr.Accordion('Advanced', open=False):
                tokens = gr.Number(value=0, label='Target output length in audio tokens (0 = auto)', precision=0)
                max_new_tokens = gr.Slider(256, 8192, value=4096, step=64, label='Max new tokens (safety cap)', info="Maximum audio length to generate. 0 = until the model says EOS.")
            btn = gr.Button('Generate speech', variant='primary', size='lg')
        with gr.Column():
            output_audio = gr.Audio(label='Generated speech', type='filepath')
            status = gr.Markdown('')
    gr.Examples(
        examples=[[e['text'], e['lang'], e['ref']] for e in EXAMPLES],
        inputs=[text, language, ref_audio],
        label='Preloaded examples (click to load)',
    )
    gr.Markdown('**Tips:** For non-Chinese/English text, **always set the language tag** — it materially improves quality. Pinyin (`ni3 hao3`) and IPA (`/h\u0259lo\u028a/`) inputs let you override pronunciation. The `[pause X.Ys]` marker inserts a precise silence.')
    btn.click(
        ui_synth,
        inputs=[text, language, ref_audio, tokens, max_new_tokens],
        outputs=[output_audio, status],
    )

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ MOSS-TTS v1.5 ready. Supports 31 languages + IPA / Pinyin / [pause X.Ys] markers.", outputs=[status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single TTS call from the notebook. Useful for scripting or quick checks.

TEXT = "We stand on the threshold of the AI era." # @param {type:"string"}
LANGUAGE = "English" # @param ["", "Chinese", "English", "Cantonese", "Arabic", "Czech", "Danish", "Dutch", "Finnish", "French", "German", "Greek", "Hebrew", "Hindi", "Hungarian", "Italian", "Japanese", "Korean", "Macedonian", "Malay", "Persian", "Polish", "Portuguese", "Romanian", "Russian", "Spanish", "Swahili", "Swedish", "Tagalog", "Thai", "Turkish", "Vietnamese"]
REF_AUDIO = "" # @param {type:"string"}
TOKENS = 0 # @param {type:"integer"}
MAX_NEW_TOKENS = 4096 # @param {type:"integer"}

path, sr = synth(
    TEXT,
    language=LANGUAGE,
    ref_audio_path=REF_AUDIO or None,
    tokens=TOKENS if TOKENS > 0 else None,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f'[Done] {path} ({sr} Hz)')
from IPython.display import Audio, display
display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each line in `BATCH` becomes one output file. **One script per line.**

BATCH_LANGUAGE = 'English' # @param ["", "Chinese", "English", "Cantonese", "Arabic", "Czech", "Danish", "Dutch", "Finnish", "French", "German", "Greek", "Hebrew", "Hindi", "Hungarian", "Italian", "Japanese", "Korean", "Macedonian", "Malay", "Persian", "Polish", "Portuguese", "Romanian", "Russian", "Spanish", "Swahili", "Swedish", "Tagalog", "Thai", "Turkish", "Vietnamese"]
BATCH_REF_AUDIO = '' # @param {type:"string"}
BATCH_TOKENS = 0 # @param {type:"integer"}
BATCH_MAX_NEW_TOKENS = 4096 # @param {type:"integer"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

import shutil
lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)

for i, line in enumerate(lines):
    try:
        print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
        path, sr = synth(
            line,
            language=BATCH_LANGUAGE,
            ref_audio_path=BATCH_REF_AUDIO or None,
            tokens=BATCH_TOKENS if BATCH_TOKENS > 0 else None,
            max_new_tokens=BATCH_MAX_NEW_TOKENS,
            out_name=f'{i:03d}.wav',
        )
        shutil.move(path, os.path.join(out_dir, f'{i:03d}.wav'))

    except Exception as e:
        print(f"  -> EXCEPTION: {e}")
        continue
print(f'\n[Done] {len(lines)} files written to {out_dir}')